# Match-IA Regional - Sprint 1
**Proyecto:** Match-IA Regional (TECH-MACI/2026-G03)  
**Objetivo del Sprint 1:** Analizar la oferta de titulados y los sectores productivos de las regiones seleccionadas, identificando patrones y definiendo indicadores relevantes para Match-IA Regional.  
**Responsable Tarea #8:** Marian García  
**Rama:** `marian/tarea-configurar-entorno-y-estructura-del-proyecto`


## Sección 0: Setup y Rutas (Funcional)
- Configuración de rutas relativas con `pathlib.Path` respecto al directorio de datos.
- Definición de rutas a `data/oferta/`, `data/geo/`, `data/demanda/`, `data/processed/`.
- Verificación y descompresión automática del dataset de titulados 2025 si no se encuentra en disco.


In [1]:
import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

# Configuración de rutas relativas usando pathlib.Path
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
else:
    PROJECT_ROOT = CURRENT_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
OFERTA_DIR = DATA_DIR / "oferta"
GEO_DIR = DATA_DIR / "geo"
DEMANDA_DIR = DATA_DIR / "demanda"
PROCESSED_DIR = DATA_DIR / "processed"

print(f"Directorio raíz del proyecto: {PROJECT_ROOT.resolve()}")
print(f"Ruta datos oferta: {OFERTA_DIR.resolve()}")
print(f"Ruta datos geo: {GEO_DIR.resolve()}")
print(f"Ruta datos procesados: {PROCESSED_DIR.resolve()}")

# Verificación y mecanismo de descompresión automática del dataset de titulados
CSV_FILENAME = "20260817_Titulados_Ed_Superior_2025_WEB.csv"
RAR_FILENAME = "Titulados-Ed-Superior-2025.rar"

csv_path = OFERTA_DIR / CSV_FILENAME
csv_subdir_path = OFERTA_DIR / "Titulados-Ed-Superior-2025" / CSV_FILENAME
rar_path = OFERTA_DIR / RAR_FILENAME

if csv_subdir_path.exists():
    titulados_csv_path = csv_subdir_path
    print(f"Dataset encontrado en subcarpeta: {titulados_csv_path.name}")
elif csv_path.exists():
    titulados_csv_path = csv_path
    print(f"Dataset encontrado en: {titulados_csv_path.name}")
else:
    print(f"Archivo descomprimido no encontrado. Procediendo a descomprimir desde {rar_path.name}...")
    target_dir = OFERTA_DIR / "Titulados-Ed-Superior-2025"
    target_dir.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(["tar", "-xf", str(rar_path), "-C", str(target_dir)], check=True)
        titulados_csv_path = target_dir / CSV_FILENAME
        print(f"Descompresión completada exitosamente: {titulados_csv_path}")
    except Exception as e:
        print(f"Aviso: no fue posible descomprimir automáticamente ({e}).")
        print(f"Por favor descomprima manualmente '{rar_path}' en '{target_dir}'.")
        titulados_csv_path = None


Directorio raíz del proyecto: C:\Users\maria\OneDrive\Desktop\Proyecto Ingeniería\2026-G03
Ruta datos oferta: C:\Users\maria\OneDrive\Desktop\Proyecto Ingeniería\2026-G03\data\oferta
Ruta datos geo: C:\Users\maria\OneDrive\Desktop\Proyecto Ingeniería\2026-G03\data\geo
Ruta datos procesados: C:\Users\maria\OneDrive\Desktop\Proyecto Ingeniería\2026-G03\data\processed
Dataset encontrado en subcarpeta: 20260817_Titulados_Ed_Superior_2025_WEB.csv


## Sección 1: Carga y Diagnóstico de Titulados (Funcional)
- Lectura optimizada del archivo CSV seleccionando únicamente las columnas útiles para el análisis.
- Reporte de dimensiones (filas y columnas), tipos de datos y % de valores nulos.
- Validación de duplicados por la combinación `mrun` + `cod_carrera` + `anio_tit`.


In [2]:
# Inspección de cabeceras disponibles en el archivo
header_sample = pd.read_csv(titulados_csv_path, sep=";", encoding="latin-1", nrows=5)
all_columns = header_sample.columns.tolist()

# Columnas útiles especificadas para el análisis
cols_to_load = [
    "mrun", "gen_alu", "fec_nac_alu", "anio_ing_carr_ori", "sem_ing_carr_ori",
    "cod_inst", "nomb_inst", "tipo_inst_1", "cod_sede", "nomb_sede",
    "cod_carrera", "nomb_carrera", "nivel_global", "nivel_carrera_1",
    "cine_f_13_area", "cine_f_13_subarea", "region_sede"
]

date_cols = [c for c in ["cat_periodo", "fecha_obtencion_titulo", "anio_tit", "mes_tit"] if c in all_columns]
cols_to_use = [c for c in cols_to_load if c in all_columns] + date_cols

print(f"Cargando {len(cols_to_use)} columnas relevantes de {len(all_columns)} totales...")
# Se utiliza encoding UTF-8 / latin-1 con fallback para evitar problemas de tildes
try:
    df_titulados = pd.read_csv(titulados_csv_path, sep=";", encoding="utf-8", usecols=cols_to_use, low_memory=False)
except UnicodeDecodeError:
    df_titulados = pd.read_csv(titulados_csv_path, sep=";", encoding="latin-1", usecols=cols_to_use, low_memory=False)

# Homologación de columnas anio_tit y mes_tit en caso de venir como cat_periodo / fecha_obtencion_titulo
if "anio_tit" not in df_titulados.columns and "cat_periodo" in df_titulados.columns:
    df_titulados["anio_tit"] = df_titulados["cat_periodo"]
if "mes_tit" not in df_titulados.columns and "fecha_obtencion_titulo" in df_titulados.columns:
    df_titulados["mes_tit"] = df_titulados["fecha_obtencion_titulo"].astype(str).str[4:6].astype(int)

n_rows, n_cols = df_titulados.shape
mem_mb = df_titulados.memory_usage(deep=True).sum() / (1024 ** 2)
print("=" * 60)
print("DIAGNÓSTICO DEL DATASET DE TITULADOS")
print("=" * 60)
print(f"Dimensiones: {n_rows:,} filas × {n_cols} columnas")
print(f"Uso de memoria en RAM: {mem_mb:.2f} MB")

null_report = pd.DataFrame({
    "Tipo de Dato": df_titulados.dtypes,
    "Valores No Nulos": df_titulados.notnull().sum(),
    "Valores Nulos": df_titulados.isnull().sum(),
    "% Nulos": (df_titulados.isnull().sum() / n_rows * 100).round(2)
})
display(null_report)

# Validación de duplicados por mrun + cod_carrera + anio_tit
dup_mask = df_titulados.duplicated(subset=["mrun", "cod_carrera", "anio_tit"], keep=False)
n_dups = df_titulados.duplicated(subset=["mrun", "cod_carrera", "anio_tit"]).sum()
pct_dups = (n_dups / n_rows) * 100
print(f"\nValidación de Duplicados (mrun + cod_carrera + anio_tit):")
print(f"- Registros duplicados detectados: {n_dups:,} ({pct_dups:.3f}%)")
if n_dups > 0:
    print("Muestra de casos duplicados identificados:")
    display(df_titulados[dup_mask].sort_values(["mrun", "cod_carrera"]).head(6)[
        ["mrun", "cod_carrera", "nomb_carrera", "cod_inst", "nomb_inst", "anio_tit"]
    ])


Cargando 19 columnas relevantes de 41 totales...
DIAGNÓSTICO DEL DATASET DE TITULADOS
Dimensiones: 328,998 filas × 21 columnas
Uso de memoria en RAM: 239.69 MB


,Tipo de Dato,Valores No Nulos,Valores Nulos,% Nulos
cat_periodo,int64,328998,0,0.00
mrun,float64,328028,970,0.29
gen_alu,int64,328998,0,0.00
fec_nac_alu,int64,328998,0,0.00
anio_ing_carr_ori,int64,328998,0,0.00
sem_ing_carr_ori,int64,328998,0,0.00
fecha_obtencion_titulo,int64,328998,0,0.00
tipo_inst_1,str,328998,0,0.00
cod_inst,int64,328998,0,0.00
nomb_inst,str,328998,0,0.00



Validación de Duplicados (mrun + cod_carrera + anio_tit):
- Registros duplicados detectados: 847 (0.257%)
Muestra de casos duplicados identificados:


,mrun,cod_carrera,nomb_carrera,cod_inst,nomb_inst,anio_tit
1481,114793.0,NaN,DIPLOMADO EN DIRECCION ESTRATEGICA,31,UNIVERSIDAD AUTONOMA DE CHILE,2025
1482,114793.0,NaN,DIPLOMADO EN ENTORNO ESTRATEGICO,31,UNIVERSIDAD AUTONOMA DE CHILE,2025
2059,158624.0,NaN,DIPLOMADO EN DIRECCION ESTRATEGICA,31,UNIVERSIDAD AUTONOMA DE CHILE,2025
2060,158624.0,NaN,DIPLOMADO EN GESTION DE RECURSOS PUBLICOS,31,UNIVERSIDAD AUTONOMA DE CHILE,2025
4417,350232.0,NaN,DIPLOMADO EN GERENCIA PARA INSTITUCIONES DE SALUD,4,UNIVERSIDAD CENTRAL DE CHILE,2025
4418,350232.0,NaN,DIPLOMADO EN GESTION SANITARIA,4,UNIVERSIDAD CENTRAL DE CHILE,2025


## Sección 2: Limpieza, Estandarización y Gratuidad (Funcional)
- Limpieza de textos y espacios en nombres de instituciones y carreras.
- Estandarización de `region_sede` a las **16 regiones oficiales de Chile** con orden geográfico Norte a Sur.
- Creación de variable booleana `area_afin_ia` según área formativa CINE-F-13.
- Cruce (Merge) con `instituciones_gratuidad_2025.csv` mediante `cod_inst` para bandera `inst_gratuidad`.
- Verificación de consistencia del cruce institucional y reporte de cobertura.


In [3]:
# 1. Función para normalización y reparación de caracteres (tildes, ñ y codificación)
def normalizar_texto(texto):
    if not isinstance(texto, str):
        return texto
    s = texto.strip()
    # Reparar mojibake común en caso de doble codificación (UTF-8 leído como latin-1)
    try:
        if "Ã" in s or "Â" in s:
            s = s.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
    import re
    return re.sub(r"\s+", " ", s)

# Aplicar normalización a las columnas de texto
text_cols = ["nomb_inst", "nomb_sede", "nomb_carrera", "tipo_inst_1", "region_sede"]
for col in text_cols:
    if col in df_titulados.columns:
        df_titulados[col] = df_titulados[col].apply(normalizar_texto)

# 2. Estandarización de las 16 regiones oficiales de Chile con orden geográfico Norte a Sur
REGIONES_NORTE_A_SUR = [
    "Arica y Parinacota",
    "Tarapacá",
    "Antofagasta",
    "Atacama",
    "Coquimbo",
    "Valparaíso",
    "Metropolitana de Santiago",
    "Libertador General Bernardo O'Higgins",
    "Maule",
    "Ñuble",
    "Biobío",
    "La Araucanía",
    "Los Ríos",
    "Los Lagos",
    "Aysén del General Carlos Ibáñez del Campo",
    "Magallanes y de la Antártica Chilena"
]

# Diccionario de homologación limpio: sin caracteres corruptos ni mojibake,
# con soporte para tildes oficiales, 'ñ', nombres oficiales completos y alias sin tilde
region_mapping = {
    # 1. Arica y Parinacota
    "Arica y Parinacota": "Arica y Parinacota",
    "Región de Arica y Parinacota": "Arica y Parinacota",
    
    # 2. Tarapacá
    "Tarapacá": "Tarapacá",
    "Tarapaca": "Tarapacá",
    "Región de Tarapacá": "Tarapacá",
    
    # 3. Antofagasta
    "Antofagasta": "Antofagasta",
    "Región de Antofagasta": "Antofagasta",
    
    # 4. Atacama
    "Atacama": "Atacama",
    "Región de Atacama": "Atacama",
    
    # 5. Coquimbo
    "Coquimbo": "Coquimbo",
    "Región de Coquimbo": "Coquimbo",
    
    # 6. Valparaíso
    "Valparaíso": "Valparaíso",
    "Valparaiso": "Valparaíso",
    "Región de Valparaíso": "Valparaíso",
    
    # 7. Metropolitana
    "Metropolitana": "Metropolitana de Santiago",
    "Metropolitana de Santiago": "Metropolitana de Santiago",
    "Región Metropolitana": "Metropolitana de Santiago",
    "Región Metropolitana de Santiago": "Metropolitana de Santiago",
    
    # 8. Libertador General Bernardo O'Higgins
    "Lib. Gral. B. O'Higgins": "Libertador General Bernardo O'Higgins",
    "Libertador General Bernardo O'Higgins": "Libertador General Bernardo O'Higgins",
    "Región del Libertador Gral. Bernardo O'Higgins": "Libertador General Bernardo O'Higgins",
    "O'Higgins": "Libertador General Bernardo O'Higgins",
    
    # 9. Maule
    "Maule": "Maule",
    "Región del Maule": "Maule",
    
    # 10. Ñuble
    "Ñuble": "Ñuble",
    "Nuble": "Ñuble",
    "Región de Ñuble": "Ñuble",
    
    # 11. Biobío
    "Biobío": "Biobío",
    "Biobio": "Biobío",
    "Región del Biobío": "Biobío",
    
    # 12. La Araucanía
    "La Araucanía": "La Araucanía",
    "La Araucania": "La Araucanía",
    "Araucanía": "La Araucanía",
    "Región de La Araucanía": "La Araucanía",
    
    # 13. Los Ríos
    "Los Ríos": "Los Ríos",
    "Los Rios": "Los Ríos",
    "Región de Los Ríos": "Los Ríos",
    
    # 14. Los Lagos
    "Los Lagos": "Los Lagos",
    "Región de Los Lagos": "Los Lagos",
    
    # 15. Aysén del General Carlos Ibáñez del Campo
    "Aysén": "Aysén del General Carlos Ibáñez del Campo",
    "Aysen": "Aysén del General Carlos Ibáñez del Campo",
    "Aysén del General Carlos Ibáñez del Campo": "Aysén del General Carlos Ibáñez del Campo",
    "Región de Aysén": "Aysén del General Carlos Ibáñez del Campo",
    
    # 16. Magallanes y de la Antártica Chilena
    "Magallanes": "Magallanes y de la Antártica Chilena",
    "Magallanes y de la Antártica Chilena": "Magallanes y de la Antártica Chilena",
    "Región de Magallanes": "Magallanes y de la Antártica Chilena"
}

df_titulados["region_sede_std"] = df_titulados["region_sede"].map(region_mapping).fillna(df_titulados["region_sede"])
df_titulados["region_sede_std"] = pd.Categorical(
    df_titulados["region_sede_std"],
    categories=REGIONES_NORTE_A_SUR,
    ordered=True
)

# 3. Creación de variable booleana area_afin_ia
AREAS_AFINES_IA = [
    "tecnología de la información y la comunicación (tic)",
    "ingeniería, industria y construcción",
    "ciencias naturales, matemáticas y estadística"
]

area_norm = df_titulados["cine_f_13_area"].astype(str).str.strip().str.lower()
df_titulados["area_afin_ia"] = area_norm.isin(AREAS_AFINES_IA)

print("Distribución de Titulados según Afinidad a IA:")
display(df_titulados["area_afin_ia"].value_counts(dropna=False).to_frame(name="Total Titulados"))

# 4. Cruce (Merge) con instituciones de gratuidad 2025
gratuidad_path = OFERTA_DIR / "instituciones_gratuidad_2025.csv"
df_gratuidad = pd.read_csv(gratuidad_path, sep=";", encoding="utf-8")

df_titulados["cod_inst"] = pd.to_numeric(df_titulados["cod_inst"], errors="coerce")
df_gratuidad["cod_inst"] = pd.to_numeric(df_gratuidad["cod_inst"], errors="coerce")

df_titulados = df_titulados.merge(
    df_gratuidad[["cod_inst", "adscrita_gratuidad"]],
    on="cod_inst",
    how="left"
)

df_titulados["inst_gratuidad"] = df_titulados["adscrita_gratuidad"].fillna(0).astype(int)
df_titulados.drop(columns=["adscrita_gratuidad"], inplace=True)

# 5. Reporte de consistencia y cobertura del cruce
inst_total = df_titulados["cod_inst"].nunique()
inst_adscritas = df_titulados[df_titulados["inst_gratuidad"] == 1]["cod_inst"].nunique()
tit_con_grat = (df_titulados["inst_gratuidad"] == 1).sum()
pct_con_grat = (tit_con_grat / len(df_titulados)) * 100

print("=" * 60)
print("REPORTE DE CRUCE CON INSTITUCIONES CON GRATUIDAD 2025")
print("=" * 60)
print(f"- Instituciones totales en titulados: {inst_total}")
print(f"- Instituciones adscritas a gratuidad: {inst_adscritas} ({inst_adscritas/inst_total*100:.1f}%)")
print(f"- Titulados en instituciones con gratuidad: {tit_con_grat:,} ({pct_con_grat:.2f}%)")
print(f"- Titulados afines a IA con gratuidad: {len(df_titulados[(df_titulados['area_afin_ia']) & (df_titulados['inst_gratuidad'] == 1)]):,}")


Distribución de Titulados según Afinidad a IA:


,Total Titulados
area_afin_ia,
False,249968
True,79030


REPORTE DE CRUCE CON INSTITUCIONES CON GRATUIDAD 2025
- Instituciones totales en titulados: 123
- Instituciones adscritas a gratuidad: 65 (52.8%)
- Titulados en instituciones con gratuidad: 200,013 (60.79%)
- Titulados afines a IA con gratuidad: 56,321


## Sección 3: Lectura Geográfica de Sedes (Funcional)
- Lectura directa del Shapefile comprimido en `.zip` con `geopandas.read_file("zip://...")`.
- Verificación y reproyección a CRS WGS84 (`EPSG:4326`).
- Conteo de sedes por región y contraste comparativo con los datos de oferta.


In [4]:
# 1. Lectura del Shapefile comprimido
geo_zip_path = GEO_DIR / "layer_establecimientos_de_educacion_superior_20220309024111(1).zip"
uri = f"zip://{geo_zip_path.resolve()}"

print(f"Cargando capas geográficas desde: {geo_zip_path.name}...")
gdf_sedes = gpd.read_file(uri)

# 2. Verificación y reproyección a WGS84 (EPSG:4326)
if gdf_sedes.crs != "EPSG:4326":
    gdf_sedes = gdf_sedes.to_crs("EPSG:4326")
print(f"CRS verificado: {gdf_sedes.crs}")
print(f"Total de sedes georreferenciadas: {len(gdf_sedes):,}")

# Mapeo de regiones por código oficial COD_REGION (1 a 16)
cod_region_to_std = {
    1.0: "Tarapacá",
    2.0: "Antofagasta",
    3.0: "Atacama",
    4.0: "Coquimbo",
    5.0: "Valparaíso",
    6.0: "Libertador General Bernardo O'Higgins",
    7.0: "Maule",
    8.0: "Biobío",
    9.0: "La Araucanía",
    10.0: "Los Lagos",
    11.0: "Aysén del General Carlos Ibáñez del Campo",
    12.0: "Magallanes y de la Antártica Chilena",
    13.0: "Metropolitana de Santiago",
    14.0: "Los Ríos",
    15.0: "Arica y Parinacota",
    16.0: "Ñuble"
}

gdf_sedes["region_std"] = gdf_sedes["COD_REGION"].astype(float).map(cod_region_to_std)
gdf_sedes["region_std"] = pd.Categorical(
    gdf_sedes["region_std"],
    categories=REGIONES_NORTE_A_SUR,
    ordered=True
)

# 3. Conteo de sedes por región y contraste con oferta de titulados
sedes_conteo = gdf_sedes.groupby("region_std", observed=False).size().rename("Sedes Físicas (Geo)")
titulados_conteo = df_titulados.groupby("region_sede_std", observed=False).size().rename("Titulados Totales")
titulados_ia_conteo = df_titulados[df_titulados["area_afin_ia"]].groupby("region_sede_std", observed=False).size().rename("Titulados Afines IA")

resumen_regional = pd.concat([sedes_conteo, titulados_conteo, titulados_ia_conteo], axis=1)
resumen_regional["% Titulados Afines IA"] = (resumen_regional["Titulados Afines IA"] / resumen_regional["Titulados Totales"] * 100).round(2)

print("\n" + "=" * 75)
print("COMPARACIÓN TERRITORIAL: SEDES VS OFERTA DE TITULADOS (NORTE A SUR)")
print("=" * 75)
display(resumen_regional)


Cargando capas geográficas desde: layer_establecimientos_de_educacion_superior_20220309024111(1).zip...
CRS verificado: EPSG:4326
Total de sedes georreferenciadas: 1,297

COMPARACIÓN TERRITORIAL: SEDES VS OFERTA DE TITULADOS (NORTE A SUR)


,Sedes Físicas (Geo),Titulados Totales,Titulados Afines IA,% Titulados Afines IA
Arica y Parinacota,30,3352,804,23.99
Tarapacá,27,3668,944,25.74
Antofagasta,44,7770,3251,41.84
Atacama,23,2973,1126,37.87
Coquimbo,43,9581,2803,29.26
Valparaíso,161,26605,7612,28.61
Metropolitana de Santiago,526,187496,41012,21.87
Libertador General Bernardo O'Higgins,27,5916,1846,31.20
Maule,65,12898,2653,20.57
Ñuble,29,5162,1124,21.77


## Sección 4: Cálculo de Indicadores de Oferta 
*Espacio reservado para implementación de indicadores de concentración, especialización y oferta académica regional.*

## Sección 5: Visualizaciones y EDA de Oferta
*Espacio reservado para gráficos exploratorios, mapas temáticos y análisis territorial de la oferta.*

## Sección 6: Ingesta de Datos de Demanda y Exposición a IA 
*Espacio reservado para la carga de datos de sectores económicos, matrices de empleo y exposición a IA.*

## Sección 7: Cálculo de Indicadores de Demanda 
*Espacio reservado para los índices de demanda laboral y necesidades de habilidades complementarias a la IA.*

## Sección 8: Matriz y Cuadrantes Match-IA 
*Espacio reservado para el cálculo de brechas, matriz de cuadrantes y priorización de regiones.*

## Sección 9: Ficha Metodológica y Exportación 
*Espacio reservado para la exportación de datasets intermedios/finales a data/processed/ y documentación metodológica.*